# 端到端提交格式 Mock

仅供检查平台导入、评测入口、输出列、采样时间和股票覆盖，不是已训练的参赛模型，也不承诺通过完整模型材料／方法合规审核。

- 仅上传本 Notebook；平台调用 `main(datasources, start_date, end_date)`。
- 返回 Pandas DataFrame，恰好包含 `date`（datetime64[ns]）、`instrument`（字符串）、`score`（float64）。
- 只读取官方允许的 `cpt_jyc_2026_instruments` 的时间与股票键；不读取未来收益标签，不加载离线预测文件。
- 采样点和股票集合直接采用评测区间内股票池的记录；不手工拼接日历，不根据本地数据假定云端范围。
- mock 分数是键和固定 seed 的确定性映射；非恒定、有限，并且同一键在不同查询区间中得到同一分数。分数没有预测含义。
- 日期形式的结束参数包含当天；带时分秒的结束参数按精确包含上界处理。有时区的参数转为 UTC+08:00。
- 依赖云端已有的 `dai`、`pandas`、`numpy`，以及 Python 标准库；不安装包、不访问外网、不读取凭据或本地行情。
- Notebook 只有函数定义，没有顶层查询或训练；由平台调用后才查询数据。没有训练脚本或权重。
- 2026-10-09（UTC+08:00）复核：[比赛介绍](https://bigquant.com/square/competition/45b41c7a-0ac8-42e9-9fe3-0c33dcfb9bca) 要求 `date, instrument, score`；[新手入门](https://bigquant.com/wiki/doc/oSGtQiGtq0) 明确入口和股票池表。


In [ ]:
def main(datasources, start_date, end_date):
    """仅验证提交格式：读取授权股票池键，生成可重复的 mock 分数。"""
    import hashlib
    import re
    from datetime import date as date_type, datetime

    import dai
    import numpy as np
    import pandas as pd

    # 官方指南允许直接使用此股票池表。本 mock 不查询行情表，
    # 因此不使用 datasources 中的 bar1m/bar5m/bar15m/bar30m。
    pool_table = "cpt_jyc_2026_instruments"
    seed = 20261009

    def as_local_time(value):
        timestamp = pd.Timestamp(value)
        if pd.isna(timestamp):
            raise ValueError("评测时间不能为空。")
        if timestamp.tzinfo is not None:
            timestamp = timestamp.tz_convert("Asia/Shanghai").tz_localize(None)
        return timestamp

    start = as_local_time(start_date)
    end = as_local_time(end_date)
    end_is_date_only = (
        isinstance(end_date, str)
        and re.fullmatch(r"\d{4}-\d{2}-\d{2}", end_date.strip()) is not None
    ) or (isinstance(end_date, date_type) and not isinstance(end_date, datetime))

    # 日期形式的 end_date 包含当天；带时分秒的 end_date 为精确的包含上界。
    # 输出键必须整分钟，因此将查询边界转换为等价的整分钟半开区间。
    if end_is_date_only:
        stop = end.normalize() + pd.Timedelta(days=1)
        if start >= stop:
            raise ValueError("start_date 必须早于 end_date 的次日零点。")
    else:
        if start > end:
            raise ValueError("start_date 不能晚于 end_date。")
        stop = end.floor("min") + pd.Timedelta(minutes=1)
    cursor = start.ceil("min")
    if cursor >= stop:
        raise ValueError("指定区间内没有完整的分钟采样时刻。")

    parts = []
    while cursor < stop:
        next_month = cursor.normalize().replace(day=1) + pd.offsets.MonthBegin(1)
        chunk_stop = min(next_month, stop)
        lower_text = cursor.strftime("%Y-%m-%d %H:%M:%S")
        upper_text = chunk_stop.strftime("%Y-%m-%d %H:%M:%S")
        sql = f"""
            SELECT DISTINCT date, instrument
            FROM {pool_table}
            WHERE date >= '{lower_text}' AND date < '{upper_text}'
            ORDER BY date, instrument
        """
        # filters 用整日边界覆盖 SQL 区间；最终边界由 SQL 严格限定，
        # 避免将末日 filters 上界设在当天零点，漏掉该日全部截面。
        keys = dai.query(
            sql,
            filters={
                "date": [
                    cursor.normalize().strftime("%Y-%m-%d"),
                    chunk_stop.ceil("D").strftime("%Y-%m-%d"),
                ]
            },
        ).df()
        if not {"date", "instrument"}.issubset(keys.columns):
            raise ValueError("股票池查询必须返回 date 和 instrument。")
        if not keys.empty:
            keys = keys.loc[:, ["date", "instrument"]].copy()
            keys["date"] = pd.to_datetime(keys["date"], errors="raise")
            if keys["date"].dt.tz is not None:
                keys["date"] = (
                    keys["date"].dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
                )
            keys["date"] = keys["date"].astype("datetime64[ns]")
            if keys[["date", "instrument"]].isna().any().any():
                raise ValueError("股票池包含空键，停止生成结果。")
            if not keys["date"].eq(keys["date"].dt.floor("min")).all():
                raise ValueError("股票池包含非整分钟时间，需要核对官方采样规则。")
            if not (keys["date"].ge(cursor) & keys["date"].lt(chunk_stop)).all():
                raise ValueError("股票池查询返回了本次 SQL 区间以外的数据。")
            keys["instrument"] = keys["instrument"].astype(str)
            if keys["instrument"].str.strip().eq("").any():
                raise ValueError("股票池包含空股票代码。")
            if keys.duplicated(["date", "instrument"]).any():
                raise ValueError("股票池查询返回了重复键。")

            # 固定 seed + 每一行自己的键；不依赖查询顺序、区间长度或标签。
            # 非恒定分数用于避免全零截面令相关性/标准化无法计算。
            timestamps = keys["date"].dt.strftime("%Y-%m-%dT%H:%M:%S")

            def mock_scores():
                for timestamp, instrument in zip(timestamps, keys["instrument"]):
                    payload = f"{seed}|{timestamp}|{instrument}".encode("utf-8")
                    digest = hashlib.blake2b(payload, digest_size=8).digest()
                    integer = int.from_bytes(digest, "big") >> 11
                    yield integer * (2.0 / (1 << 53)) - 1.0

            keys["score"] = np.fromiter(
                mock_scores(), dtype=np.float64, count=len(keys)
            )
            parts.append(keys)
        cursor = chunk_stop

    if not parts:
        raise ValueError(
            "评测区间内股票池为空：请核对比赛空间、数据权限和评测日期。"
        )

    result = pd.concat(parts, ignore_index=True)
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)
    if result.duplicated(["date", "instrument"]).any():
        raise ValueError("跨分区结果包含重复键。")
    if not np.isfinite(result["score"].to_numpy()).all():
        raise ValueError("score 必须全部为有限数值。")
    counts = result.groupby("date", sort=False)["score"].agg(["size", "nunique"])
    if ((counts["size"] > 1) & (counts["nunique"] < 2)).any():
        raise ValueError("发现多股票截面的 mock 分数完全相同。")
    print(
        f"FORMAT MOCK ONLY | rows={len(result)} | "
        f"timestamps={len(counts)} | seed={seed}"
    )
    return result[["date", "instrument", "score"]]
